# Fase IV: Selección del Modelo Final, Evaluación en Test Set y Prototipo Funcional

**Proyecto Integrador - Inteligencia Computacional**  
**Dominio:** Medio Ambiente &rarr; Calidad del Agua (*Water Quality*)  
**Integrantes:** Melanie Tomalá, Jean Cedeño, Damian Torres Cadena  
**Docente:** Ing. Daniel Quirumbay  
**Institución:** Universidad Estatal Península de Santa Elena (UPSE) &mdash; 2026-1  

---

## Objetivos de la Fase IV (según rúbrica oficial de cátedra)
1. **Seleccionar UN modelo final:** fundamentar la elección según métricas de validación cruzada (Macro-F1, ROC-AUC, Recall), estabilidad, complejidad computacional y requisitos del problema ambiental, argumentando el descarte de los demás.
2. **Reentrenar el modelo seleccionado:** utilizar la totalidad del conjunto de entrenamiento (`data/processed/train.csv`, 6.396 registros) y validar **estrictamente una sola vez** en el conjunto de prueba reservado (`data/processed/test.csv`, 1.600 registros).
3. **Serializar el pipeline completo:** guardar el artefacto entrenado con preprocesamiento anti-fuga (`models/final_pipeline.joblib`).
4. **Demostración de 3 casos de uso:** evaluar perfiles representativos (agua potable, agua contaminada y caso límite/adverso) verificando el manejo elegante de datos inválidos y comparación contra umbrales de la OMS.
5. **Despliegue de Prototipo:** integrar la inferencia en una interfaz funcional en Streamlit (`src/water_quality_app.py`).

## 1. Configuración del Entorno y Carga de Artefactos

In [ ]:
import os
from pathlib import Path
if not Path('data').exists() and Path('../data').exists():
    os.chdir('..')
import os
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import Image, display

# Configurar estilo visual
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (9, 6)
plt.rcParams["font.size"] = 11

print("Entorno inicializado correctamente.")

## 2. Selección final

GridSearch obtuvo Macro-F1 CV 0,8979 ± 0,0119 frente a 0,8967 ± 0,0132 de RF-GA. Se conservó RF-GA para evidenciar el eje metaheurístico, por decisión documentada antes del test. No se afirma superioridad ni equivalencia estadística. Las cifras se leen del JSON.


In [ ]:
# Cargar resultados consolidados de Fase III para verificación
with open('reports/phase3_results.json', 'r', encoding='utf-8') as f:
    phase3_data = json.load(f)

summary_rows = []
for model_name, metrics in phase3_data["models"].items():
    summary_rows.append({
        "Modelo": model_name,
        "Macro-F1 (CV)": f"{metrics['macro_f1_mean']:.4f} ± {metrics['macro_f1_std']:.4f}",
        "ROC-AUC (CV)": f"{metrics['roc_auc_mean']:.4f} ± {metrics['roc_auc_std']:.4f}",
        "Recall Segura (1)": f"{metrics['recall_mean']:.4f}",
        "Accuracy (CV)": f"{metrics['accuracy_mean']:.4f}",
        "Tiempo Fit (s)": f"{metrics['fit_time_mean']:.2f}"
    })

df_comparison = pd.DataFrame(summary_rows)
display(df_comparison)

## 3. Verificación del Pipeline Serializado y Protocolo Anti-Fuga

Para garantizar la reproducibilidad y el despliegue desacoplado, el pipeline empaqueta:
1. `StandardScaler`: ajustado con media y varianza aprendidas **exclusivamente del conjunto de entrenamiento** (6.396 registros).
2. `RandomForestClassifier`: configurado con los hiperparámetros óptimos encontrados por el Algoritmo Genético.

In [ ]:
# Cargar pipeline serializado
pipeline = joblib.load('models/final_pipeline.joblib')
print("Tipo de artefacto:", type(pipeline))
print("Etapas del pipeline:", [name for name, _ in pipeline.steps])

# Inspeccionar hiperparámetros del clasificador final
rf_model = pipeline.named_steps['model']
print("\nHiperparámetros del Random Forest final:")
for param in ['n_estimators', 'max_depth', 'min_samples_split', 'min_samples_leaf', 'criterion', 'class_weight']:
    print(f"  - {param}: {getattr(rf_model, param)}")

## 4. Lectura de la evaluación final ya guardada

La evaluación predictiva se realizó una sola vez. La exploración previa del conjunto completo y la inspección del encabezado y una fila de test se declaran como limitaciones de sellado. Esta sección solo lee el JSON persistido; no abre test.csv ni vuelve a evaluar.


In [ ]:
# Cargar el reporte oficial de evaluación de Fase IV
with open('reports/phase4_selection.json', 'r', encoding='utf-8') as f:
    phase4_rep = json.load(f)

tm = phase4_rep["test_metrics"]
cm = tm["confusion_matrix_labels_0_1"]

print("=== MÉTRICAS OFICIALES EN TEST SET RESERVADO (1.600 MUESTRAS) ===")
print(f"Macro-F1:                 {tm['macro_f1']:.4f} ({tm['macro_f1']*100:.2f}%)")
print(f"ROC-AUC:                  {tm['roc_auc']:.4f}")
print(f"Accuracy:                 {tm['accuracy']:.4f} ({tm['accuracy']*100:.2f}%)")
print(f"Recall (Clase 1 Segura):  {tm['recall_class_1_safe']:.4f} ({tm['recall_class_1_safe']*100:.2f}%)")
print(f"Recall (Clase 0 No Seg.): {tm['recall_class_0_unsafe']:.4f} ({tm['recall_class_0_unsafe']*100:.2f}%)")
print(f"Precision (Clase 1):      {tm['precision_class_1_safe']:.4f}")
print(f"Precision (Clase 0):      {tm['precision_class_0_unsafe']:.4f}")

print("\nMatriz de Confusión:")
print(f"  [TN={cm[0][0]:4d} | FP={cm[0][1]:3d}]  (Clase 0: No Segura / Contaminada)")
print(f"  [FN={cm[1][0]:4d} | TP={cm[1][1]:3d}]  (Clase 1: Segura según dataset)")

In [ ]:
# Visualizar la figura generada de la matriz de confusión en test set
fig_path = 'reports/figures/08_test_confusion_matrix.png'
if os.path.exists(fig_path):
    display(Image(filename=fig_path))
else:
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=["Pred Contaminada (0)", "Pred Segura (1)"],
                yticklabels=["Real Contaminada (0)", "Real Segura (1)"])
    plt.title("Matriz de Confusión - Test Set Reservado")
    plt.tight_layout()
    plt.show()

## 5. Casos de demostración

A: perfil de train de clase segura, sin certificación sanitaria. B: metales y nitrato alto con extrapolación. C: falla adversa conocida (clase 1 pese a alertas y valores fuera de train), que debe marcar conflicto. Las comparaciones OMS requieren confirmar mg/L y especie química.


In [ ]:
from backend.api import get_presets, WaterSampleInput, predict_water_quality

presets = get_presets()
for case in presets.values():
    result = predict_water_quality(WaterSampleInput(**case["data"]))
    print(case["name"], result.label, "Conflicto:", result.conflict)
    print("Alertas condicionales:", result.who_alerts)
    print("Fuera de train:", result.out_of_range_features)
    print(result.interpretation_note)


### Manejo de Entradas Inválidas y Outliers Físicos

El prototipo implementa validaciones estrictas previas al cómputo: rechazo de valores negativos, caracteres no numéricos o ausencia de campos requeridos.

In [ ]:
from pydantic import ValidationError

invalid = presets["case_a_safe"]["data"].copy()
invalid["ammonia"] = -0.08
try:
    WaterSampleInput(**invalid)
except ValidationError as error:
    print("Entrada rechazada antes de inferir:", error.errors()[0]["msg"])


## 6. Prototipo FastAPI y React

Consultar el README raíz. Compilar con `npm --prefix frontend ci` y `npm --prefix frontend run build`. Ejecutar `python -m uvicorn backend.api:app --host 127.0.0.1 --port 8000`. Abrir http://127.0.0.1:8000. Modificar un valor y analizar un registro nuevo durante la defensa. No ejecutar de nuevo la evaluación final.
